In [ ]:
import pandas as pd
import seaborn as sb 
import matplotlib.pyplot as plt
import os,sys
import pandas as pd
from matplotlib.lines import Line2D
import matplotlib as mpl
import numpy as np
import pingouin as pg
from scipy.stats import zscore
sys.path.extend(['/Users/amonast/Documents/GitHub/Engram_2P/Engram_2P'])
from rois.rois import remove_bad_cells
plt.style.use('paper_style.mplstyle')

savepath = '/Users/amonast/Documents/GitHub/CA1_Engram_Dynamics/figures'

# Calculate mCherry intensities.

In [ ]:
base_dir = '/Volumes/AM_SSD1/Spont2P'
file_key = '/Volumes/AM_SSD1/Spont2P/Data_info.csv'
animals = ['589L',
            '989N',
            '992N',
            '992L',
            '994R',
            '9972R',
            '217R',
            '217N',
            '218L',
            '034R',
            '149L',
            '146R',
            '160R',
            '492N',
            '493R',
            '1912L']
fov_lists = [['FOV1','FOV2'],
            ['FOV1','FOV2'],
            ['FOV2'],
            ['FOV2'],
            ['FOV1','FOV2'],
            ['FOV1','FOV2'],
            ['FOV1'],
            ['FOV1','FOV2'],
            ['FOV1','FOV2'],
            ['FOV1'],
            ['FOV1','FOV2'],
            ['FOV2'],
            ['FOV1','FOV2'],
            ['FOV1','FOV2'],
            ['FOV1'],
            ['FOV1','FOV2']]

info = pd.read_csv(file_key)
DF = pd.DataFrame()
for a,ani in enumerate(animals):
    fovs = fov_lists[a]
    for fov in fovs:
        ind_df = remove_bad_cells(ani,fov,file_key,base_dir,snr_thr=4.0,filter_mchleak=True)
        ind_df = ind_df.loc[(ind_df.Baseline!=-1) & (ind_df.Post!=-1)]
        group = info.Group.loc[info.Animal==ani].values[0]
        ind_df['FOV']=[fov]*len(ind_df)
        ind_df['Animal']=[ani]*len(ind_df)
        ind_df['Group'] = [group]*len(ind_df)
        zscored=zscore(np.concatenate((ind_df.mch_pre,ind_df.mch_post)))
        ind_df['mch_Z_pre']= zscored[0:int(len(zscored)/2)]
        ind_df['mch_Z_post']= zscored[0:int(len(zscored)/2)]
        DF = pd.concat([DF,ind_df],ignore_index=True)

# mch_pre = [float(DF['mch_pre'].values[i][1:-1]) for i in range(len(DF))]
# mch_post =  [float(DF['mch_post'].values[i][1:-1]) for i in range(len(DF))]

DF_reg1 = DF.loc[(DF.Baseline!=-1)&(DF.Post!=-1)]

## Save CSV

In [ ]:
DF.to_csv('mcherry_intensities_all.csv')

## Make dataframe

In [ ]:
DF_reg1=DF.copy()
DF_reg = pd.concat([DF_reg1.loc[:,['Baseline','Post','Tagged','Animal','FOV','Group']],DF_reg1.loc[:,['Baseline','Post','Tagged','Animal','FOV','Group']]],ignore_index=True)
DF_reg['Session'] = ['Day 0']*len(DF_reg1)+['Day 4']*len(DF_reg1)
ints = np.concatenate((DF_reg1.mch_pre.values,DF_reg1.mch_post.values))
zscored=np.concatenate((DF_reg1.mch_Z_pre.values,DF_reg1.mch_Z_post.values))
DF_reg['mCherryInt'] = ints
DF_reg['ZScoredInt']=zscored
DF_reg['ZScoreAllCell']=zscore(DF_reg.mCherryInt)
DF_reg

# Supp Fig 1C 

In [ ]:
plt.figure(figsize=(2,2))
ax=plt.gca()
g=sb.histplot(data=DF_reg,x='mCherryInt',log_scale=True,hue='Session',ax=ax,stat='proportion',common_norm=False,kde=True,hue_order=['Day 4', 'Day 0'],palette={'firebrick','thistle'})
ax.set_xlim([0,500])
ax.set_xlabel('mCherry Intensity (AU)',fontdict={'size':8})
ax.set_ylabel('Fraction of Cells',fontdict={'size':10})
plt.yticks(fontsize=8)
plt.xticks(fontsize=8)
sb.despine()
lines = [Line2D([0], [0], color='thistle', lw=2),
        Line2D([0], [0], color='firebrick', lw=2)]
ax.legend(lines,['Day 0','Day 4'],bbox_to_anchor=(.3,.8),frameon=False,fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(savepath,'mch_intensity_regcells_all_hist.svg'),dpi=300,transparent=True)

In [ ]:
plt.figure(figsize=(3,3))
ax=plt.gca()
g=sb.histplot(data=DF_reg,x='ZScoreAllCell',log_scale=True,hue='Session',ax=ax,stat='proportion',common_norm=False,kde=True,hue_order=['Day 4', 'Day 0'],palette={'firebrick','thistle'})
ax.set_xlabel('mCherry Intensity (AU)',fontdict={'size':8})
ax.set_ylabel('Fraction of Cells',fontdict={'size':10})
plt.xlim([-2,5])
plt.yticks(fontsize=8)
plt.xticks(fontsize=8)
sb.despine()
lines = [Line2D([0], [0], color='thistle', lw=2),
        Line2D([0], [0], color='firebrick', lw=2)]
ax.legend(lines,['Day 0','Day 4'],bbox_to_anchor=(.3,.8),frameon=False,fontsize=8)
plt.tight_layout()

In [ ]:
pg.wilcoxon(DF_reg.mCherryInt.loc[(DF_reg.Session=='Day 0')].values,
       DF_reg.mCherryInt.loc[(DF_reg.Session=='Day 4')].values)

# Supp Fig 1D

In [ ]:
plt.figure(figsize=(2,2))
ax=plt.gca()
g=sb.kdeplot(data=DF_reg.loc[(DF_reg.Session=='Day 4')],x='mCherryInt',log_scale=True,hue='Tagged',ax=ax,common_norm=False,hue_order=[0,1],palette=['#00ABC8','#F37243'])
# ax.set_xlim([0,200])
# ax.set_ylim([0,.05])

ax.set_xlabel('mCherry Intensity (AU)',fontdict={'size':8})
ax.set_ylabel('Fraction of Cells',fontdict={'size':10})
plt.yticks(fontsize=10)
plt.xticks(fontsize=10)
sb.despine()
lines = [Line2D([0], [0], color='#00ABC8', lw=2),
        Line2D([0], [0], color='#F37243', lw=2)]
ax.legend(lines,['Non-engram','Engram'],bbox_to_anchor=(1.9,1),loc='upper right',frameon=False,fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(savepath,'mch_intensityD4_regcells_split_kde_log.svg'),dpi=300,transparent=True)

In [ ]:
pg.normality(data=DF_reg.loc[DF_reg.Session=='Day 4'],dv='mCherryInt',group='Tagged')

In [ ]:
pg.mwu(DF_reg.mCherryInt.loc[(DF_reg.Session=='Day 4')&(DF_reg.Tagged==1)].values,
       DF_reg.mCherryInt.loc[(DF_reg.Session=='Day 4')&(DF_reg.Tagged==0)].values)

# Supp Figure 1G

In [ ]:
fold_df = pd.DataFrame()
for ani in animals:
    df_ani = DF_reg.loc[DF_reg['Animal']==ani].reset_index(drop=True)
    tagged = df_ani['Tagged'].iloc[:int(df_ani.shape[0]/2)]
    group = [df_ani.Group.values[0]]*int(df_ani.Group.shape[0]/2)
    fold_change = df_ani['mCherryInt'].loc[df_ani['Session']=='Day 4'].values/df_ani['mCherryInt'].loc[df_ani['Session']=='Day 0'].values
    fold = pd.DataFrame()
    fold['Fold Change mCherry']=fold_change
    fold['ZScored Fold Change']=zscore(fold_change)
    fold['Animal']=[ani]*fold_change.shape[0]
    fold['Group']=group
    fold['Tagged']=tagged
    mean_fold = fold['Fold Change mCherry'].mean()
    fold['Normalized Fold Change mCherry'] = fold['Fold Change mCherry']/mean_fold
    fold['Normalized mCherry Int']  = df_ani['mCherryInt']/df_ani['mCherryInt'].loc[df_ani.Session=='Day 0'].mean()
    fold_df = pd.concat([fold_df,fold],ignore_index=True)

In [ ]:

plt.figure(figsize=(2.5,2))
sb.boxplot(data=fold_df,y='Fold Change mCherry',hue='Tagged',order=['FC','HC'],x='Group',palette=['#00ABC8','#F37243'],fliersize=2)
sb.despine()
plt.gca().get_legend().remove()
plt.hlines(9.7,-.3,.1,color='k')
plt.text(-.1,9.8,s='****',size=10,ha='center')
plt.hlines(11.7,.8,1.2,color='k')
plt.text(1,11.9,s='****',size=10,ha='center')


plt.savefig(os.path.join(savepath,'mch_foldchange_regcells_splitboxplots.svg'),dpi=300,transparent=True)

In [ ]:

plt.figure(figsize=(2.5,2))
sb.boxplot(data=fold_df,y='ZScored Fold Change',hue='Tagged',order=['FC','HC'],x='Group',palette=['#00ABC8','#F37243'],showfliers=True)
sb.despine()
plt.gca().get_legend().remove()
# plt.hlines(9.7,-.3,.1,color='k')
# plt.text(-.1,9.8,s='****',size=10,ha='center')
# plt.hlines(11.7,.8,1.2,color='k')
# plt.text(1,11.9,s='****',size=10,ha='center')



In [ ]:
pg.normality(data=fold_df,dv='Fold Change mCherry',group='Group')

In [ ]:
pg.anova(data=fold_df,dv='Fold Change mCherry',between=['Group','Tagged'])

In [ ]:
pg.pairwise_tests(data=fold_df,dv='Fold Change mCherry',between=['Group','Tagged'],padjust='bonf',parametric=False)

In [ ]:
ani_fold = fold_df.groupby(['Group','Animal','Tagged']).mean().reset_index()

# Supplementary Figure 1H

In [ ]:
mpl.rcParams['font.family']='Galvji'
mpl.rcParams['svg.fonttype']='none'
plt.figure(figsize=(1.8,2))
#sb.stripplot(data=ani_fold,y='Fold Change mCherry',hue='Tagged',order=['FC','HC'],x='Group',palette=['gray','gray'],dodge=True)
sb.boxplot(data=ani_fold,y='ZScored Fold Change',hue='Tagged',x='Group',palette=['#00ABC8','#F37243'],dodge=True)
plt.hlines(1,-.3,.1,color='k')
plt.text(-.1,1,s='****',size=10,ha='center')
plt.hlines(1.4,.8,1.2,color='k')
plt.text(1,1.4,s='****',size=10,ha='center')
sb.despine()
plt.gca().get_legend().remove()


plt.savefig(os.path.join(savepath,'mch_foldchange_animals_splitboxplot_zscore.svg'),dpi=300,transparent=True)

In [ ]:
pg.anova(data=ani_fold,dv='ZScored Fold Change',between=['Group','Tagged'])

In [ ]:
pg.pairwise_tests(data=ani_fold,dv='ZScored Fold Change',between=['Group','Tagged'],padjust='holm')